# 🌌 QuATernaryons: Sub-2-Bit MiniCPM Evaluation Benchmark
### Dual-Sided Sub-2-Bit Weight & Dynamic KV-Cache Quantization via 4D Isoclinic Clifford Rotors

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23530592.svg)](https://doi.org/10.5281/zenodo.23530592)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Solstice-Labs/QuATernaryons/blob/main/QuATernaryons_MiniCPM_Benchmark.ipynb)

---

## 📐 Mathematical Advantage: Why MiniCPM > SmolLM
- **MiniCPM Head Dimension:** $d_h = 64 = 4^3$ — an **exact power of 4**!
- **Rotor Decompositions:** Each head consists of exactly $16$ parallel $4\times 4$ quaternion rotors across $\log_4(64) = 3$ butterfly stages.
- **Zero Padding:** Every dimension in MiniCPM ($d=2304 = 36\times 64$, $d_{\text{mlp}} = 5760 = 90\times 64$) splits cleanly into 64-channel blocks. Zero padding, zero dead SIMD lanes, 100% compute efficiency.
- **SmolLM Comparison:** SmolLM2-1.7B has $d=2048 = 2^{11}$, which is *not* a power of 4, requiring hybrid stages or padding.

## 🎯 The Zero-Data Evaluation Gate (The One Metric That Matters)
- **Objective:** Measure $\mathrm{KL}(P_{\text{teacher}} \parallel P_{\text{student}})$ in **nats/token** and **Top-1 Agreement** on inputs drawn from the model's own distribution.
- **Sample Protocol:** Exactly 65,536 teacher-generated tokens, sequence length 1024, random seed 1234, top-k 40 continuations.
- **Near-Lossless Bar:** $\mathrm{KL} \le 0.05$ nats/token ($\approx 5\%$ perplexity inflation).
- **Autoregressive Coherence Guard:** Tests greedy generation to verify no MC-generation dissociation / rotor phase drift.

In [ ]:
# @title 1. GPU Check & Environment Setup
!nvidia-smi
!pip install -q torch transformers accelerate safetensors

In [ ]:
# @title 2. Clone QuATernaryons Engine
!git clone https://github.com/Solstice-Labs/QuATernaryons.git /content/QuATernaryons
%cd /content/QuATernaryons

In [ ]:
# @title 3. Run MiniCPM Sub-2-Bit Benchmark (PTQ + Zero-Data Polish)
!python3 colab_minicpm_quaternaryons.py \
    --model-id openbmb/MiniCPM-2B-sft-bf16 \
    --group-size 32 \
    --rank 8 \
    --n-eval-tokens 65536 \
    --seq-len 1024 \
    --device cuda \
    --do-qat \
    --qat-steps 300 \
    --output-json /content/minicpm_quaternaryons_results.json

In [ ]:
# @title 4. Display Benchmark Results & Comparison Table
import json
from IPython.display import display, Markdown

with open('/content/minicpm_quaternaryons_results.json') as f:
    res = json.load(f)

md = f"""
### 🏆 QuATernaryons Sub-2-Bit Distribution Fidelity Results

| Pipeline Stage | Model | Bit-Rate (bpw) | KL (nats/token) | Top-1 Agreement | Lossless Gate (<0.05) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **SmolLM2 Baseline (PTQ)** | SmolLM2-1.7B | 2.16 | 0.2951 | 69.61% | ❌ Miss |
| **SmolLM2 Baseline (QAT)** | SmolLM2-1.7B | 2.16 | 0.1337 | 79.95% | ❌ Miss |
| **MiniCPM (PTQ)** | {res['model_id']} | {res['avg_bpw']:.2f} | **{res['ptq']['kl_nats_per_token']:.4f}** | **{res['ptq']['top1_agreement_pct']:.2f}%** | {'✅ PASS' if res['ptq']['kl_nats_per_token'] <= 0.05 else '⚡ Frontier'} |
"""
if res.get('qat'):
    md += f"| **MiniCPM (QAT Polish)** | {res['model_id']} | {res['avg_bpw']:.2f} | **{res['qat']['kl_nats_per_token']:.4f}** | **{res['qat']['top1_agreement_pct']:.2f}%** | {'✅ PASS' if res['qat']['kl_nats_per_token'] <= 0.05 else '⚡ Frontier'} |\n"

display(Markdown(md))

print("\n--- Sample Autoregressive Generation ---")
for prompt, gen in list(res['coherence_samples'].items())[:2]:
    print(f"\nPrompt: {prompt}")
    print(f"Response:\n{gen}")
